In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../Outputs/merged_text_905.csv")
print(f"Loaded: {df.shape[0]} rows")
print(f"Posts with usable text: {df['has_text'].sum()}")

Loaded: 905 rows
Posts with usable text: 797


In [3]:
MODEL_NAME = "MMADS/MoralFoundationsClassifier"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()

# Get label names from model config
labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
print(f"Number of labels: {len(labels)}")
print(f"Labels: {labels}")

Using device: cpu


Loading weights: 100%|██████████| 201/201 [00:05<00:00, 36.18it/s]


Number of labels: 10
Labels: ['LABEL_0', 'LABEL_1', 'LABEL_2', 'LABEL_3', 'LABEL_4', 'LABEL_5', 'LABEL_6', 'LABEL_7', 'LABEL_8', 'LABEL_9']


In [4]:
from tqdm import tqdm

texts = df.loc[df["has_text"], "text_for_analysis"].tolist()
texts = [t[:1500] for t in texts]

batch_size = 16
all_scores = []

print(f"Running MMADS on {len(texts)} posts...")
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    # Multi-label sigmoid, not softmax (each label is independent)
    probs = torch.sigmoid(outputs.logits).cpu().numpy()
    all_scores.extend(probs.tolist())

print(f"Done. Got {len(all_scores)} score sets")

Running MMADS on 797 posts...


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [07:57<00:00,  9.56s/it]

Done. Got 797 score sets


In [5]:
mor_cols = [f"mmads_{label}" for label in labels]

# Drop existing columns if they exist
df = df.drop(columns=[c for c in mor_cols + ["mmads_dominant", "mmads_dominant_score"] if c in df.columns])

# Build score dataframe
scores_df = pd.DataFrame(all_scores, columns=labels)
scores_df.columns = mor_cols
scores_df.index = df.index[df["has_text"]]
df = df.join(scores_df)

# Compute dominant foundation
df["mmads_dominant"] = None
df["mmads_dominant_score"] = None
df.loc[df["has_text"], "mmads_dominant"] = (
    df.loc[df["has_text"], mor_cols].idxmax(axis=1).str.replace("mmads_", "")
)
df.loc[df["has_text"], "mmads_dominant_score"] = df.loc[df["has_text"], mor_cols].max(axis=1)

print(df[["student_id", "text_source", "mmads_dominant", "mmads_dominant_score"]].head())

  student_id         text_source mmads_dominant mmads_dominant_score
0        1_A  caption+transcript        LABEL_6             0.941466
1        1_A  caption+transcript        LABEL_0             0.971412
2        1_A        caption_only        LABEL_6             0.002599
3        2_A        caption_only        LABEL_6             0.006314
4        2_A        caption_only        LABEL_6             0.006473


In [6]:
print("=== Dominant foundation counts ===")
print(df["mmads_dominant"].value_counts(dropna=False))
print()
print("=== Mean scores across all posts ===")
print(df[mor_cols].mean().sort_values(ascending=False))

=== Dominant foundation counts ===
mmads_dominant
LABEL_6    539
None       108
LABEL_0    101
LABEL_4     62
LABEL_8     43
LABEL_1     22
LABEL_2     13
LABEL_9     13
LABEL_7      3
LABEL_3      1
Name: count, dtype: int64

=== Mean scores across all posts ===
mmads_LABEL_0    0.168166
mmads_LABEL_6    0.108666
mmads_LABEL_4    0.103219
mmads_LABEL_8    0.080231
mmads_LABEL_1    0.059549
mmads_LABEL_9    0.038992
mmads_LABEL_2    0.033425
mmads_LABEL_3    0.009329
mmads_LABEL_7    0.006121
mmads_LABEL_5    0.002292
dtype: float64


In [7]:
print("=== Dominant foundation by text source ===")
print(pd.crosstab(df["text_source"], df["mmads_dominant"]))
print()
print("=== Mean dominant score by text source ===")
print(df.groupby("text_source")["mmads_dominant_score"].agg(["mean", "median", "max", "count"]))

=== Dominant foundation by text source ===
mmads_dominant      LABEL_0  LABEL_1  LABEL_2  LABEL_3  LABEL_4  LABEL_6  \
text_source                                                                
caption+transcript       36       13        5        0       19       50   
caption_only             64        9        8        1       43      489   
transcript_only           1        0        0        0        0        0   

mmads_dominant      LABEL_7  LABEL_8  LABEL_9  
text_source                                    
caption+transcript        0       14        6  
caption_only              3       29        6  
transcript_only           0        0        1  

=== Mean dominant score by text source ===
                        mean    median       max  count
text_source                                            
caption+transcript  0.706231  0.955189  0.997142    143
caption_only        0.274718  0.006167   0.99535    652
none                     NaN       NaN       NaN      0
transcript_o

In [8]:
print("=== Per-student dominant foundation breakdown ===")
print(pd.crosstab(df["student_id"], df["mmads_dominant"]))

=== Per-student dominant foundation breakdown ===
mmads_dominant  LABEL_0  LABEL_1  LABEL_2  LABEL_3  LABEL_4  LABEL_6  LABEL_7  \
student_id                                                                      
10_A                  5        2        0        0        5       42        0   
11_A                 18        3        4        0       16      126        0   
12_A                  5        2        0        0        4       51        0   
1_A                   4        2        1        1        0       47        0   
2_A                   8        0        1        0        3       44        0   
3_A                  17        0        5        0       12       25        0   
4_A                   9        1        0        0        1       39        1   
5_A                   8       10        2        0       12       36        0   
6_A                   6        1        0        0        1       34        0   
7_A                   1        0        0        0        0

In [9]:
# Show top 2 examples per foundation
for foundation in sorted(df["mmads_dominant"].dropna().unique()):
    subset = df[df["mmads_dominant"] == foundation]
    if len(subset) < 3:
        continue
    print(f"\n=== {foundation.upper()} ({len(subset)} posts) ===")
    top = subset.nlargest(2, f"mmads_{foundation}")
    for _, r in top.iterrows():
        text_preview = r['text_for_analysis'][:200].replace('\n', ' | ')
        print(f"  [{r[f'mmads_{foundation}']:.2f}] [{r['text_source']}] {text_preview}")


=== LABEL_0 (101 posts) ===
  [1.00] [caption+transcript] Tai Lung and Po edit x DUCKWORTH #fyp #fyp? #xyzbca #kungfupanda #edit |  | You take two strangers and put them in random predicaments Give them a soul so they can make their own choices and live with it 
  [0.99] [caption_only] Thousands of worshippers flocked to Hong Kongs best-known Taoist temple on Monday for the annual ritual of the burning of the first incense sticks, marking the start of Lunar New Year with prayers fo

=== LABEL_1 (22 posts) ===
  [0.99] [caption+transcript] The Greatest Double Agent Ever |  | There really is nothing quite like the element of surprise. When the Allied forces appeared over the horizon on the morning of D-Day, it completely caught the Nazis off
  [0.99] [caption_only] Concerned fans of USA skater llia | Malinin flooded his social media with messages of support after the Winter Olympics star shared some vulnerable posts following his nightmare competing on Friday nigh

=== LABEL_2 (13 post

In [10]:
# Q9 has moral categories. Let's see if MMADS dominant aligns with student-reported Q9 categories
q9_col = "Q9. What MORAL/IMMORAL ideas did the post seem to express? (Select all that apply)"
df_with_q9 = df[df[q9_col].notna() & df["has_text"]]
print(f"Posts with both Q9 response AND model output: {len(df_with_q9)}")
print()
print("=== Cross-tab: MMADS dominant foundation vs first Q9 category ===")
# Take just the first comma-separated entry of Q9
df_with_q9 = df_with_q9.copy()
df_with_q9["q9_first"] = df_with_q9[q9_col].astype(str).str.split(',').str[0].str.strip()
print(pd.crosstab(df_with_q9["mmads_dominant"], df_with_q9["q9_first"]))

Posts with both Q9 response AND model output: 185

=== Cross-tab: MMADS dominant foundation vs first Q9 category ===
q9_first        Authority (respect for rules  Betrayal (disloyalty  \
mmads_dominant                                                       
LABEL_0                                    0                     0   
LABEL_1                                    0                     0   
LABEL_2                                    0                     0   
LABEL_4                                    0                     0   
LABEL_6                                    3                     1   
LABEL_7                                    0                     0   
LABEL_8                                    0                     0   
LABEL_9                                    0                     0   

q9_first        Care (protecting people  Corruption / Degradation  Curiosity  \
mmads_dominant                                                                 
LABEL_0               

In [11]:
# Inspect the model config more carefully
print("id2label:", model.config.id2label)
print()
print("label2id:", model.config.label2id)
print()
# Check if there's a problem_type or task description
print("problem_type:", getattr(model.config, "problem_type", "not set"))

id2label: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5', 6: 'LABEL_6', 7: 'LABEL_7', 8: 'LABEL_8', 9: 'LABEL_9'}

label2id: {'LABEL_0': 0, 'LABEL_1': 1, 'LABEL_2': 2, 'LABEL_3': 3, 'LABEL_4': 4, 'LABEL_5': 5, 'LABEL_6': 6, 'LABEL_7': 7, 'LABEL_8': 8, 'LABEL_9': 9}

problem_type: None


In [12]:
# Rename LABEL_X to proper moral foundation names
label_mapping = {
    "mmads_LABEL_0": "mmads_care_virtue",
    "mmads_LABEL_1": "mmads_care_vice",
    "mmads_LABEL_2": "mmads_fairness_virtue",
    "mmads_LABEL_3": "mmads_fairness_vice",
    "mmads_LABEL_4": "mmads_loyalty_virtue",
    "mmads_LABEL_5": "mmads_loyalty_vice",
    "mmads_LABEL_6": "mmads_authority_virtue",
    "mmads_LABEL_7": "mmads_authority_vice",
    "mmads_LABEL_8": "mmads_sanctity_virtue",
    "mmads_LABEL_9": "mmads_sanctity_vice"
}
df = df.rename(columns=label_mapping)

# Recompute dominant with new names
mor_cols = list(label_mapping.values())
df["mmads_dominant"] = None
df["mmads_dominant_score"] = None
df.loc[df["has_text"], "mmads_dominant"] = (
    df.loc[df["has_text"], mor_cols].idxmax(axis=1).str.replace("mmads_", "")
)
df.loc[df["has_text"], "mmads_dominant_score"] = df.loc[df["has_text"], mor_cols].max(axis=1)

print("=== Dominant foundation counts (with proper names) ===")
print(df["mmads_dominant"].value_counts(dropna=False))

=== Dominant foundation counts (with proper names) ===
mmads_dominant
authority_virtue    539
None                108
care_virtue         101
loyalty_virtue       62
sanctity_virtue      43
care_vice            22
fairness_virtue      13
sanctity_vice        13
authority_vice        3
fairness_vice         1
Name: count, dtype: int64


In [13]:
output_path = "../Outputs/morality_MMADS_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../Outputs/morality_MMADS_905.csv


# Morality Model 1: MMADS MoralFoundationsClassifier — Conclusion

**Model:** `MMADS/MoralFoundationsClassifier`
**Output labels:** 10 — five moral foundations × virtue and vice (care/harm, fairness/cheating, loyalty/betrayal, authority/subversion, sanctity/degradation)
**Trained on:** Personal blogs, political blogs, news articles, essays, parliamentary debates, UN speeches, central bank speeches
**Dataset:** 905 posts from 12 students, 797 had usable text

## What we did in plain terms

Ran every post through a single multi-label classifier that scores each of 10 moral foundation dimensions independently. Unlike MFormer (which uses 5 separate binary models), this is one model that outputs all 10 scores at once.

## What we found

| Foundation | Count | Share |
|---|---|---|
| authority_virtue | 539 | 68% |
| care_virtue | 101 | 13% |
| loyalty_virtue | 62 | 8% |
| sanctity_virtue | 43 | 5% |
| care_vice | 22 | 3% |
| fairness_virtue | 13 | 2% |
| sanctity_vice | 13 | 2% |
| authority_vice | 3 | <1% |
| fairness_vice | 1 | <1% |
| loyalty_vice | 0 | 0% |

## The problem

Two issues showed up clearly.

**First, the model card explicitly warns against using this model on social media.** From the official documentation: "It may not accurately generalize to texts that are significantly different in style or domain from its training data (NOT RECOMMENDED FOR SOCIAL MEDIA DATA)." MMADS was trained on parliamentary debates, news articles, and formal essays. Not social media posts.

**Second, the output reflects this mismatch.** 68% of posts were tagged as authority_virtue, which is unrealistic for typical student social media feeds. The model is likely defaulting to authority/tradition framing because that's what dominates its training data. Vice categories barely appeared at all (only 39 total vice posts out of 797).

## Useful aspects we did get

The model does correctly identify clear cases:
- Caring prayers ("I pray your mind, your spirit, and your heart find healing") → care_virtue at 0.99
- D-Day double agent stories → care_vice (harm) at 0.99
- Health equity goals → fairness_virtue at 0.99
- Family celebrations → loyalty_virtue at 0.99

When the post is clearly about a moral theme, the model gets it right. The problem is the catch-all behavior on ambiguous content getting dumped into authority_virtue.

## Comparison with Q9 student-reported categories

Q9 has a similar virtue/vice taxonomy. Cross-tabulating MMADS dominant labels with Q9 first-listed categories on the 185 posts with both:

The model's authority_virtue label captured almost every Q9 category — including 46 posts students labeled as Care, 13 as Loyalty, 19 as Purity, 13 as Harm. The model is not differentiating well at all.

## Bottom line

MMADS is the wrong tool for this dataset. The authors of the model explicitly warn against its use on social media, and our results confirm that warning. 68% of posts dumped into authority_virtue with no real differentiation.

We're keeping this output as documentation of "what doesn't work" but should not rely on these labels for any analysis. MFormer (next) is trained on a broader text mix and should be more reliable. The dictionary method (eMFD) will provide a transparent baseline that we can compare against.

The takeaway for the project: model selection for morality detection on social media is harder than emotion or toxicity. Most pretrained moral classifiers are trained on formal text and don't generalize to casual posts.